In [ ]:
# ============================================================
# RESONANCE ANALYSIS NOTEBOOK (FULL PUBLICATION VERSION)
# ============================================================

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve
from scipy import stats

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.grid"] = True

RESULTS_DIR = "results"
DATASET = "imdb"   # change if needed

# ------------------------------------------------------------
# Load Results
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json")))
print("Found", len(files), "result files")

raw = []
for fp in files:
    with open(fp, "r", encoding="utf-8") as f:
        raw.append(json.load(f))

# ------------------------------------------------------------
# Build DataFrame
# ------------------------------------------------------------

rows = []

for j in raw:
    meta = j.get("meta", {})
    test = j.get("best_metrics", {}).get("test", {})

    rows.append({
        "dataset": meta.get("dataset"),
        "model": meta.get("model"),
        "seed": meta.get("seed"),
        "accuracy": test.get("accuracy"),
        "macro_f1": test.get("macro_f1"),
        "weighted_f1": test.get("weighted_f1"),
        "balanced_acc": test.get("balanced_accuracy"),
        "auroc": test.get("auroc"),
        "auprc": test.get("auprc"),
        "ece": test.get("ece"),
        "nll": test.get("nll"),
        "brier": test.get("brier"),
        "total_minutes": j.get("total_minutes"),
    })

df = pd.DataFrame(rows)
df = df[df["dataset"] == DATASET]
print(df.head())

# ------------------------------------------------------------
# Benchmark Table (mean ± std)
# ------------------------------------------------------------

print("\n===== BENCHMARK SUMMARY =====\n")
summary = df.groupby("model").agg(["mean", "std", "count"])
display(summary)

# ------------------------------------------------------------
# Pareto Plot (Accuracy vs Time)
# ------------------------------------------------------------

plt.figure()
for model, g in df.groupby("model"):
    plt.scatter(g["total_minutes"], g["accuracy"], label=model)

plt.title("Efficiency Frontier")
plt.xlabel("Total Training Time (minutes)")
plt.ylabel("Test Accuracy")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Calibration Scatter (ECE vs Accuracy)
# ------------------------------------------------------------

plt.figure()
for model, g in df.groupby("model"):
    plt.scatter(g["ece"], g["accuracy"], label=model)

plt.xlabel("ECE (lower is better)")
plt.ylabel("Accuracy")
plt.title("Calibration vs Accuracy")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Reliability Diagram
# ------------------------------------------------------------

def reliability_curve(y_true, y_prob, bins=10):
    bin_edges = np.linspace(0, 1, bins+1)
    bin_ids = np.digitize(y_prob, bin_edges) - 1
    acc = []
    conf = []
    for b in range(bins):
        idx = bin_ids == b
        if idx.sum() == 0:
            acc.append(np.nan)
            conf.append(np.nan)
        else:
            acc.append((y_true[idx] == (y_prob[idx] >= 0.5)).mean())
            conf.append(y_prob[idx].mean())
    return conf, acc

plt.figure()
for j in raw:
    meta = j.get("meta", {})
    if meta.get("dataset") != DATASET:
        continue
    y_true = j.get("best_metrics", {}).get("test", {}).get("y_true")
    y_prob = j.get("best_metrics", {}).get("test", {}).get("y_prob_pos")
    if y_true is None or y_prob is None:
        continue

    conf, acc = reliability_curve(np.array(y_true), np.array(y_prob))
    plt.plot(conf, acc, label=meta.get("model"))

plt.plot([0,1],[0,1],'--')
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability Diagram")
plt.legend()
plt.show()

# ------------------------------------------------------------
# ROC Curve
# ------------------------------------------------------------

plt.figure()
for j in raw:
    meta = j.get("meta", {})
    if meta.get("dataset") != DATASET:
        continue
    y_true = j.get("best_metrics", {}).get("test", {}).get("y_true")
    y_prob = j.get("best_metrics", {}).get("test", {}).get("y_prob_pos")
    if y_true is None or y_prob is None:
        continue

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, label=meta.get("model"))

plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# ------------------------------------------------------------
# PR Curve
# ------------------------------------------------------------

plt.figure()
for j in raw:
    meta = j.get("meta", {})
    if meta.get("dataset") != DATASET:
        continue
    y_true = j.get("best_metrics", {}).get("test", {}).get("y_true")
    y_prob = j.get("best_metrics", {}).get("test", {}).get("y_prob_pos")
    if y_true is None or y_prob is None:
        continue

    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    plt.plot(recall, precision, label=meta.get("model"))

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Resonance Delta Norm Plot
# ------------------------------------------------------------

plt.figure()
for j in raw:
    meta = j.get("meta", {})
    if meta.get("dataset") != DATASET:
        continue
    if meta.get("model") != "tmr":
        continue

    deltas = j.get("best_metrics", {}).get("test", {}).get("resonance_delta_norms")
    if deltas is None:
        continue

    x = np.arange(1, len(deltas)+1)
    plt.plot(x, deltas, marker="o", label=f"seed{meta.get('seed')}")

plt.xlabel("Settling Step")
plt.ylabel("Δ Memory Norm")
plt.title("Resonance Dynamics")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Grouped Box Plot (Ablation / Seeds)
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="model", y="accuracy")
sns.stripplot(data=df, x="model", y="accuracy", color="black", alpha=0.4)
plt.title("Accuracy Distribution Across Seeds")
plt.show()

# ------------------------------------------------------------
# Statistical Comparison (Example: TMR vs TinyTransformer)
# ------------------------------------------------------------

tmr_vals = df[df["model"]=="tmr"]["accuracy"]
tt_vals = df[df["model"]=="tiny_transformer"]["accuracy"]

if len(tmr_vals) > 1 and len(tt_vals) > 1:
    t_stat, p_val = stats.ttest_ind(tmr_vals, tt_vals)
    print("\nTMR vs TinyTransformer p-value:", p_val)
else:
    print("\nNot enough seeds for statistical test.")
